# Audit DB Intermediate: How to Slice an Unbounded Log

## Filtering, Search, and Pagination

**Difficulty: Intermediate | ~40 min | Requires Lab 3 (Querying Across the Hierarchy with JOINs) done in this Supabase project**

*Lab 4 of 8 in the Audit DB Labs module.*

Labs 1-3 taught you how to write events, model them into a hierarchy, and read them back with JOINs and aggregation. But every query so far returned the *whole* log. A real audit log grows without bound - thousands of runs, millions of events - and you almost never want all of it. You want a **slice**: only the runs that failed, only the last 24 hours, only the calls that cost more than a threshold, only the payloads that mention a suspicious string, and then only the first page of results, not the entire table.

This lab is about turning an unbounded log into a readable answer. It adds **filters** (time windows, ranges, categories), **text search** (finding a needle in a payload), and **pagination** (walking through a result set one page at a time). All of it runs against your real Supabase Postgres, so every count and timestamp you see is genuine.

This lab builds only on Labs 1-3 - a schema and JOINs. It deliberately uses nothing from Lab 7: no triggers, no roles, no hash chains. Those are later; this lab is purely about `WHERE` and `LIMIT`.

In [ ]:
!pip install python-dotenv==1.2.3 psycopg2-binary==2.9.12


One pinned line installs everything the lab needs, matching Section 9 exactly.

### Step 1 — Connect to Postgres

In [ ]:
import os
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
import psycopg2

env_path = next((p / ".env" for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()), None)
load_dotenv(env_path)
database_url = os.getenv("DATABASE_URL")
assert database_url, "DATABASE_URL not found - complete README Section 7 setup first."

connection = psycopg2.connect(database_url, connect_timeout=10)
cursor = connection.cursor()
cursor.execute("SELECT version();")
print(cursor.fetchone()[0])

Same connection pattern as Labs 1-3: walk upward for the module-level `.env`, load `DATABASE_URL`, connect with `psycopg2`, and print the version string as proof the connection works.

### Step 2 — Confirm the Schema Is Already Here

In [ ]:
# Fail-fast: confirm all prerequisite tables exist (run + event from Lab 1, the Lab 2 children)
cursor.execute("SELECT to_regclass('public.run')")
assert cursor.fetchone()[0] is not None, "run table missing - complete Lab 1 first."
cursor.execute("SELECT to_regclass('public.event')")
assert cursor.fetchone()[0] is not None, "event table missing - complete Lab 1 first."
for table in ["span", "tool_call", "guardrail_event"]:
    cursor.execute("SELECT to_regclass(%s)", (f"public.{table}",))
    assert cursor.fetchone()[0] is not None, f"{table} table missing - complete Lab 2 first."
print("Schema found: run, event, span, tool_call, guardrail_event.")

Same fail-fast pattern as Labs 2 and 3: `to_regclass` checks each table without raising if it's missing. This lab needs the `run` and `event` tables from Lab 1 (for the time/range/category filters and for text search) and the child tables from Lab 2 (the JOINs that let us find runs by what happened inside them). If any table isn't here, stop and complete the prerequisite lab first.

### Step 3 — Seed a Realistic Spread of Runs to Filter

In [ ]:
import secrets

# One tag for this whole run-through, so cleanup can find exactly these rows.
lab4_tag = secrets.token_hex(4)

# (agent_name, status, total_cost, started_a_go) -- a MIX so filtering has something to bite on.
# started_a_go is a PostgreSQL interval string relative to now(): hours = recent, days = old.
seed_runs = [
    ("support-bot",   "success", 0.0210, "2 hours"),
    ("support-bot",   "error",   0.0045, "1 hour"),
    ("billing-agent", "success", 0.0900, "30 minutes"),
    ("billing-agent", "timeout", 0.0120, "3 days"),
    ("support-bot",   "success", 0.0031, "5 days"),
    ("copilot",       "error",   0.1100, "4 hours"),
    ("copilot",       "success", 0.0077, "6 days"),
    ("support-bot",   "timeout", 0.0012, "12 hours"),
    ("billing-agent", "success", 0.4500, "2 days"),
    ("copilot",       "error",   0.0088, "45 minutes"),
]

inserted_run_ids = []
for i, (agent, status, cost, ago) in enumerate(seed_runs):
    marker = f"{lab4_tag}-{i}"
    cursor.execute(
        "INSERT INTO run (agent_name, status, total_cost, started_at, ended_at) "
        "VALUES (%s, %s, %s, now() - interval %s, now()) RETURNING run_id",
        (f"pytest-lab4-{marker}", status, cost, ago),
    )
    inserted_run_ids.append(cursor.fetchone()[0])

# Seed a few events too, so text search has payloads to look through.
# One run's event contains the marker word 'refund_query' -- searchable by ILIKE.
event_payloads = [
    "tool=query_orders status=ok",
    "tool=refund_query status=pending customer_id=9912",
    "tool=query_orders status=err",
    "tool=update_ticket status=ok",
]
for i, run_id in enumerate(inserted_run_ids[:4]):
    cursor.execute(
        "INSERT INTO event (run_id, event_type, payload) VALUES (%s, %s, %s)",
        (run_id, "tool_call", event_payloads[i]),
    )

connection.commit()
print(f"Seeded {len(inserted_run_ids)} tagged runs under marker pytest-lab4-{lab4_tag}-* "
      f"(run ids {inserted_run_ids[0]}..{inserted_run_ids[-1]}).")

The seed is a deliberately varied **testing corpus**. Some runs are marked `success`, some `error`, some `timeout`. Costs range from a fraction of a cent to nearly half a dollar. Some runs started minutes ago; others are days old. And a few have `event` rows with payloads, one of which mentions the marker word `refund_query`. That spread is the whole point: a filter is only meaningful when there's something to exclude. Without the tag in `agent_name`, cleanup couldn't tell our synthetic rows apart from real activity, and re-running the lab would accumulate garbage. The tag is what makes the lab **re-runnable** - run it a hundred times and each pass cleans up after itself.

### Step 4 — Time-Window Filtering: the Last 24 Hours

In [ ]:
# Recent-only: started in the last 24 hours.
cursor.execute(
    """SELECT count(*) FROM run
       WHERE agent_name LIKE %s
         AND started_at >= now() - interval '24 hours'""",
    (f"pytest-lab4-{lab4_tag}-%",),
)
recent_24h = cursor.fetchone()[0]

# A bounded window (a specific start and end) with BETWEEN.
cursor.execute(
    """SELECT count(*) FROM run
       WHERE agent_name LIKE %s
         AND started_at BETWEEN now() - interval '3 hours' AND now() - interval '1 hour'""",
    (f"pytest-lab4-{lab4_tag}-%",),
)
mid_window = cursor.fetchone()[0]

print(f"Runs started in the last 24 hours: {recent_24h}")
print(f"Runs started between 3h and 1h ago (BETWEEN window): {mid_window}")

Time is the axis every audit log grows along, so **time-window filters are the most common audit query**: "what failed in the last hour?", "what did the previous shift see?", "is this spike new or normal?". There are two idioms here. `started_at >= now() - interval '24 hours'` is a one-sided window - everything from a point in the past forward. `BETWEEN` is a bounded window - rows from exactly two points in time. Note how `now() - interval '3 hours'` is computed inside the query, not in Python; the database's clock is the source of truth.

The `LIKE %s` on `agent_name` filters to only our tagged rows so the count reflects just this lab's seed, not the whole table. But here's the scalability lesson worth carrying forward: a time filter over a huge table is slow unless `started_at` is indexed. The more the log grows, the more a scan of every row to find "the last 24 hours" costs. Indexing `started_at` (as a real production log would) lets Postgres jump straight to the window instead of reading everything.

### Step 5 — Range & Threshold Filtering: Cost and Combining

In [ ]:
# Threshold: only runs that cost more than $0.02.
expensive_threshold = 0.02
cursor.execute(
    """SELECT count(*) FROM run
       WHERE agent_name LIKE %s AND total_cost > %s""",
    (f"pytest-lab4-{lab4_tag}-%", expensive_threshold),
)
expensive = cursor.fetchone()[0]

# Combine predicates with AND: expensive AND errored.
cursor.execute(
    """SELECT count(*) FROM run
       WHERE agent_name LIKE %s AND total_cost > %s AND status = %s""",
    (f"pytest-lab4-{lab4_tag}-%", expensive_threshold, "error"),
)
expensive_errored = cursor.fetchone()[0]

print(f"Runs costing more than ${expensive_threshold}: {expensive}")
print(f"...and also marked error: {expensive_errored}")

A **range filter** asks "how much?" or "how long?" - cost, latency, token count - anything you can order. Here `total_cost > %s` keeps only the expensive runs. Notice the value `0.02` is passed as a **parameter** (`%s`), never written into the SQL string. That's a hard rule in this module: an audit log stores untrusted data, and the moment you interpolate a value into SQL you've opened a path for an attacker to inject their own command. Parameterized queries keep the value as data, not as code. It also happens to be cleaner - no quoting headaches.

Filters compose. `AND` narrows further: cost *and* status together. A real alerting pipeline (Lab 6's territory) builds exactly this kind of predicate: "any run that cost more than X **and** errored is worth investigating." Each `AND` is another constraint the database applies to the same set of rows.

### Step 6 — Category Filtering: One Status or a Set

In [ ]:
# Exactly one category value.
cursor.execute(
    """SELECT count(*) FROM run
       WHERE agent_name LIKE %s AND status = %s""",
    (f"pytest-lab4-{lab4_tag}-%", "success"),
)
success_count = cursor.fetchone()[0]

# A set of categories: = ANY(array).
want_statuses = ["error", "timeout"]
cursor.execute(
    """SELECT count(*) FROM run
       WHERE agent_name LIKE %s AND status = ANY(%s)""",
    (f"pytest-lab4-{lab4_tag}-%", want_statuses),
)
any_count = cursor.fetchone()[0]

print(f"Runs with status = 'success': {success_count}")
print(f"Runs with status IN (error, timeout) via = ANY(array): {any_count}")

**Categories** are discrete labels - `status`, `agent_name`, `event_type` - the opposite of ranges. `status = %s` picks one value. To pick a set, there are two spellings. `status IN ('error', 'timeout')` reads naturally but is awkward to build from a Python list (you'd have to count the `%s` placeholders and expand them). `status = ANY(%s)` takes a single array parameter - pass the whole Python list as one value. `= ANY(array)` is the idiomatic way to say "in this list" in Postgres, and it keeps exactly one parameter, which keeps the injection surface small and the code tidy.

### Step 7 — Text Search in Payloads: ILIKE vs Full-Text

In [ ]:
# Find events whose payload mentions a marker word anywhere (case-insensitive substring).
needle = "refund_query"
cursor.execute(
    """SELECT e.event_id, e.event_type, e.payload
       FROM event e
       JOIN run r ON r.run_id = e.run_id
       WHERE r.agent_name LIKE %s AND e.payload ILIKE %s
       ORDER BY e.event_id""",
    (f"pytest-lab4-{lab4_tag}-%", f"%{needle}%"),
)
ilike_hits = cursor.fetchall()

# The scalable alternative is Postgres full-text search (to_tsvector / to_tsquery);
# it needs an index to be fast at scale, but the corpus here is tiny.

# Show that a different marker word finds nothing (proves the filter actually filters).
cursor.execute(
    """SELECT count(*) FROM event e
       JOIN run r ON r.run_id = e.run_id
       WHERE r.agent_name LIKE %s AND e.payload ILIKE %s""",
    (f"pytest-lab4-{lab4_tag}-%", "%nonexistent_word%"),
)
none_hits = cursor.fetchone()[0]

print(f"Events whose payload contains '{needle}' (ILIKE):")
for event_id, event_type, payload in ilike_hits:
    print(f"  event {event_id} [{event_type}]: {payload}")
print(f"Events containing 'nonexistent_word': {none_hits}")

Text lives inside log payloads, and finding a needle in a haystack is a text problem, not a numbers problem. `ILIKE '%refund_query%'` does a **case-insensitive substring** search: it returns any payload that contains that sequence of characters anywhere. The `%` wildcards match any characters around the needle. This is simple and works anywhere, which is why it's the right first tool.

But substring search has a scaling problem: `%needle%` can't use a normal index (Postgres can't jump to "a '%' somewhere in the middle"), so a big table becomes a full scan every time. When the log is large, the scalable alternative is **full-text search**: `to_tsvector(payload)` breaks the text into searchable tokens and `to_tsquery('refund & query')` matches against them, and this *can* be indexed. Full-text also understands language (stops words, stemming) that `ILIKE` can't. The tradeoff: `ILIKE` is exact and trivial; full-text is faster at scale and smarter but needs setup. For a quick poke at a small log, `ILIKE` wins. For a production audit store, plan on full-text indexes. We join to `run` only so we can scope the search to this lab's tagged runs - without that, we'd be searching everyone's events.

### Step 8 — Pagination: OFFSET vs Keyset

In [ ]:
PAGE_SIZE = 3

# Page 1 via OFFSET (skip 0). We include started_at so it can double as the keyset key.
cursor.execute(
    """SELECT run_id, agent_name, status, started_at FROM run
       WHERE agent_name LIKE %s
       ORDER BY started_at DESC, run_id DESC
       LIMIT %s OFFSET %s""",
    (f"pytest-lab4-{lab4_tag}-%", PAGE_SIZE, 0),
)
page1 = cursor.fetchall()

cursor.execute(
    """SELECT run_id, agent_name, status, started_at FROM run
       WHERE agent_name LIKE %s
       ORDER BY started_at DESC, run_id DESC
       LIMIT %s OFFSET %s""",
    (f"pytest-lab4-{lab4_tag}-%", PAGE_SIZE, PAGE_SIZE),
)
page2 = cursor.fetchall()

# Keyset (seek) pagination: carry the last row's sort key instead of skipping.
last = page1[-1]
cursor.execute(
    """SELECT run_id, agent_name, status, started_at FROM run
       WHERE agent_name LIKE %s
         AND (started_at, run_id) < (%s, %s)
       ORDER BY started_at DESC, run_id DESC
       LIMIT %s""",
    (f"pytest-lab4-{lab4_tag}-%", last[3], last[0], PAGE_SIZE),
)
keyset_page2 = cursor.fetchall()

print(f"Page 1 (LIMIT {PAGE_SIZE} OFFSET 0):")
for run_id, agent, status, _started in page1:
    print(f"  run {run_id} | {agent} | {status}")
print(f"Page 2 (LIMIT {PAGE_SIZE} OFFSET {PAGE_SIZE}):")
for run_id, agent, status, _started in page2:
    print(f"  run {run_id} | {agent} | {status}")
print(f"Keyset page 2 (seek from run {last[0]}):")
for run_id, agent, status, _started in keyset_page2:
    print(f"  run {run_id} | {agent} | {status}")

This is the intellectual peak of the lab. A filtered slice can still be thousands of rows, so you fetch it **one page at a time**. Two strategies:

**OFFSET** is the simple reading: `LIMIT 3 OFFSET 0`, then `LIMIT 3 OFFSET 3`, then `OFFSET 6`... it skips the first *N* rows and returns the next page. It's easy to understand, but it has a **drift problem**: the database must scan and discard every skipped row on every page, so cost grows as the offset grows, and if a new row is inserted between page requests, the pages shift and you can see the same row twice or miss one entirely. Fine for a demo, bad for a live, changing log.

**Keyset (seek)** pagination fixes both. Instead of a number to skip, you carry the **last row you saw** and ask for "the next rows after it": `WHERE (started_at, run_id) < (last_time, last_id)`. The sort key points at a place in the stream rather than a count, so an insert elsewhere can't shift your position, and the database can use an index to jump straight there instead of scanning skipped rows. We use the pair `(started_at, run_id)` as the key on purpose: `started_at` alone can tie (two runs at the exact same instant), and `run_id` breaks the tie to keep the order fully deterministic. At scale, keyset is the difference between a paging query that stays milliseconds and one that degrades on every page.

### Step 9 — Cleanup and Recap

In [ ]:
# Delete only this lab's tagged rows - events first (they reference runs), then the runs.
cursor.execute(
    "DELETE FROM event WHERE run_id IN (SELECT run_id FROM run WHERE agent_name LIKE %s)",
    (f"pytest-lab4-{lab4_tag}-%",),
)
cursor.execute("DELETE FROM run WHERE agent_name LIKE %s", (f"pytest-lab4-{lab4_tag}-%",))
connection.commit()

# Verify cleanup left nothing behind.
cursor.execute("SELECT count(*) FROM run WHERE agent_name LIKE %s", (f"pytest-lab4-{lab4_tag}-%",))
remaining_runs = cursor.fetchone()[0]
cursor.execute(
    "SELECT count(*) FROM event WHERE run_id IN (SELECT run_id FROM run WHERE agent_name LIKE %s)",
    (f"pytest-lab4-{lab4_tag}-%",),
)
remaining_events = cursor.fetchone()[0]

cursor.close()
connection.close()

print(f"Cleanup done - {remaining_runs} tagged runs, {remaining_events} tagged events left behind.")
print("Lab 4 complete: time filters, ranges, categories, text search, and pagination.")

Close the connection, the same good hygiene as Labs 1-3. Cleanup runs *before* closing and verifies its own work: it deletes events first (they hold a foreign key to `run`, so they must go before the runs do) and then the runs, and finally counts how many tagged rows remain - which should be zero. This is what makes the lab re-runnable: the seed tag and this teardown bookend every pass.

What this lab proved: an unbounded log becomes a readable answer the moment you can **filter** (by time, range, or category), **search** (find a needle in a payload), and **page** (walk a slice without re-scanning everything). These aren't flourishes - they're the verbs an audit system needs before anything can be *measured*. Lab 5 takes exactly these filtered, paginated slices and turns them into metrics a dashboard reads.